# LOT-RC Forecasting Pipeline Walkthrough

This notebook isolates each stage of the pipeline described in the paper:

1. **Stage 1** — Generate particle trajectory (synthetic or real-world)
2. **Stage 2** — LOT embedding (OT plan → barycentric projection → maps → velocities)
3. **Stage 3** — RC training (teacher-forced driving → ridge regression)
4. **Stage 4** — Autonomous forecasting (closed-loop velocity prediction)
5. **Stage 5** — Integration & reconstruction (cumulative sum → predicted measures)
6. **Stage 6** — Evaluation (L²(σ) RMSE, visual comparison)

**Paper reference**: "Reservoir Computing on LOT-Embedded Measure-Valued Dynamical Systems"

### Critical: Coordinate Normalization
- **Synthetic data**: use RAW coordinates (radius 1–6 for swirling, ±1.5 for geodesic). Normalizing to [0,1]² kills the velocity advantage.
- **GOES real-world data**: use [0,1]² pixel-normalized coordinates.

In [ ]:
import sys, numpy as np, matplotlib.pyplot as plt
from pathlib import Path

REPO = Path(".").resolve()
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

# Paper parameters (§5.2)
PAPER_SR = 0.7        # spectral radius
PAPER_INPUT_SCALE = 0.1
PAPER_LEAK = 0.7      # leak rate
PAPER_RIDGE_LOT = 1e-6   # ridge for velocity (LOT)
PAPER_RIDGE_RAW = 1e-4   # ridge for positions (RAW)
PAPER_T_TRAIN = 50       # training window (§5.3)
PAPER_H = 48             # forecast horizon (§5.4)

np.random.seed(42)

---
## Stage 1: Generate Particle Trajectory

The paper benchmarks on two synthetic systems:

**Swirling Cluster** (§5.5): Breathing spiral — particles on unit circle rotate while radius oscillates from 1 to r_max=6. n_steps=250, n_cycles=2, noise σ=0.02. Cycle length = 125.

**Geodesic Transport** (§5.6): W₂ geodesic between circle (r=0.8) and equilateral triangle (R=1.0). Sinusoidal reparameterization. n_steps=400, n_cycles=2. Cycle length = 200.

In [ ]:
from data_utils.simulation.measure_dynamical_systems import (
    SwirlingClusterSystem, GeodesicTransportSimulator
)

# ── Swirling Cluster ──
N = 200  # particle count
sw = SwirlingClusterSystem(N=N, noise_scale=0.02, seed=42)
traj_swirl = sw.get_cyclical_trajectory(n_cycles=2, n_steps=250, max_radius=6.0)
print(f"Swirling cluster: {traj_swirl.shape}")  # (250, 200, 2)
print(f"  Coordinate range: [{traj_swirl.min():.2f}, {traj_swirl.max():.2f}]")
print(f"  NOT normalized — radius spans 1 to 6")

# ── Geodesic Transport ──
theta = np.linspace(0, 2*np.pi, N, endpoint=False)
source_pts = np.column_stack([np.cos(theta), np.sin(theta)]) * 0.8
v = np.array([[0, 1.0], [np.sin(2*np.pi/3), -0.5], [-np.sin(2*np.pi/3), -0.5]])
target_pts = []
per_side = [N // 3, N // 3, N - 2 * (N // 3)]
for i, ns in enumerate(per_side):
    a, b = v[i], v[(i+1) % 3]
    for t in np.linspace(0, 1, ns, endpoint=False):
        target_pts.append((1 - t) * a + t * b)
target_pts = np.array(target_pts)

geo = GeodesicTransportSimulator(source_pts, target_pts)
traj_geo = geo.get_cyclical_trajectory(n_cycles=2, n_steps=400)
print(f"\nGeodesic transport: {traj_geo.shape}")  # (400, 200, 2)
print(f"  Coordinate range: [{traj_geo.min():.2f}, {traj_geo.max():.2f}]")
print(f"  NOT normalized — spans ~[-1.5, 1.5]")

In [ ]:
# Visualize a few frames
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for row, (traj, name) in enumerate([(traj_swirl, "Swirling"), (traj_geo, "Geodesic")]):
    T = traj.shape[0]
    for col, frac in enumerate([0, 0.15, 0.3, 0.5, 0.75]):
        t = int(frac * (T - 1))
        ax = axes[row, col]
        ax.scatter(traj[t, :, 0], traj[t, :, 1], s=8, alpha=0.6)
        ax.set_title(f"{name} t={t}", fontsize=10)
        ax.set_aspect("equal")
        ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()

---
## Stage 2: LOT Embedding

For each frame t, we:
1. Fix a reference σ with R points (gaussian_iso = N(0, I₂), std=1.0)
2. Solve OT plan γ ∈ ℝ^{R×N} between σ and μ_t using EMD (or Sinkhorn ε=0.01)
3. Barycentric projection: u_t(x_i) = Σ_j y_j γ_ij / Σ_j γ_ij
4. Compute LOT velocity: Δ_t = u_{t+1} - u_t

**Output**: lot_maps (T, R, 2) and velocities (T-1, R, 2)

In [ ]:
from data_utils.simulation.generate_lot_embeddings import (
    generate_reference,
    compute_ot_plan,
    plan_to_barycentric_weights,
    apply_barycentric_projection,
)

# Pick one system to walk through
traj = traj_swirl   # or traj_geo
system_name = "Swirling Cluster"
T, N, d = traj.shape
R = N  # full resolution reference (alpha_ref = 1.0)

# 2a. Generate reference measure
sigma = generate_reference("gaussian_iso", R, trajectory=traj, d=d)
print(f"Reference σ: {sigma.shape}, range [{sigma.min():.2f}, {sigma.max():.2f}]")

# 2b. Compute LOT maps for each frame (per_frame assignment)
lot_maps = np.empty((T, R, d))
for t in range(T):
    plan = compute_ot_plan(sigma, traj[t], ot_method="emd")
    weights = plan_to_barycentric_weights(plan)
    lot_maps[t] = apply_barycentric_projection(weights, traj[t])
    if t % 50 == 0:
        print(f"  Frame {t}/{T}: plan sum={plan.sum():.4f}, map range=[{lot_maps[t].min():.3f}, {lot_maps[t].max():.3f}]")

# 2c. Compute LOT velocities
velocities = lot_maps[1:] - lot_maps[:-1]  # (T-1, R, d)

print(f"\nlot_maps: {lot_maps.shape}")
print(f"velocities: {velocities.shape}")
print(f"velocity std: {velocities.std():.6f}")

In [ ]:
# Verify LOT maps track the trajectory
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for col, frac in enumerate([0, 0.25, 0.5, 0.75]):
    t = int(frac * (T - 1))
    ax = axes[col]
    ax.scatter(traj[t, :, 0], traj[t, :, 1], s=6, alpha=0.4, c="gray", label="particles")
    ax.scatter(lot_maps[t, :, 0], lot_maps[t, :, 1], s=6, alpha=0.4, c="blue", label="LOT map")
    ax.scatter(sigma[:, 0], sigma[:, 1], s=3, alpha=0.2, c="red", label="σ ref")
    ax.set_title(f"t={t}"); ax.set_aspect("equal"); ax.grid(alpha=0.15)
    if col == 0: ax.legend(fontsize=7)
fig.suptitle(f"{system_name}: LOT maps track the particle distribution")
fig.tight_layout()
plt.show()

---
## Stage 3: RC Training (Teacher Forcing)

**VELOCITY method** (proposed): Input = Δ_t, Target = Δ_{t+1}. Ridge = 1e-6.

**POSITIONS method** (baseline): Input = u_t (LOT map), Target = u_{t+1}. Ridge = 1e-4.

Both use T_train = 50 steps of teacher forcing. The reservoir is driven with the correct inputs.

Architecture: sparse ESN, n_r = N, spectral radius 0.7, leak rate 0.7, input scaling 0.1, no bias, tanh.

In [ ]:
from rc_computer import ReservoirComputer, ReservoirConfig, InitScheme

T_TRAIN = PAPER_T_TRAIN  # 50
H = PAPER_H              # 48

# ── VELOCITY paradigm: v_t -> v_{t+1} ──
vel_flat = velocities.reshape(T - 1, R * d)  # (T-1, R*2)
vel_input = vel_flat[:-1]   # Δ_0, ..., Δ_{T-3}
vel_target = vel_flat[1:]   # Δ_1, ..., Δ_{T-2}

train_input_vel = vel_input[:T_TRAIN]
train_target_vel = vel_target[:T_TRAIN]

print(f"VELOCITY training: input {train_input_vel.shape}, target {train_target_vel.shape}")

rc_vel = ReservoirComputer(ReservoirConfig(
    input_size=R * d,
    reservoir_size=N,          # n_r = N (alpha_res = 1.0)
    output_size=R * d,
    spectral_radius=PAPER_SR,  # 0.7
    input_scaling=PAPER_INPUT_SCALE,
    leak_rate=PAPER_LEAK,      # 0.7
    ridge_param=PAPER_RIDGE_LOT,  # 1e-6
    bias_scale=0.0,            # no bias
    activation="tanh",
    init_scheme=InitScheme.SPARSE,
    sparsity=0.1,
    random_seed=42,
    use_operator_norm=False,   # scale by spectral radius, not operator norm
))

states_vel = rc_vel.run(train_input_vel)       # teacher-forced driving
rc_vel.train(states_vel, train_target_vel)      # ridge regression

train_pred_vel = rc_vel.predict(states_vel)
train_rmse_vel = np.sqrt(np.mean((train_pred_vel - train_target_vel) ** 2))
print(f"  Train RMSE (velocity): {train_rmse_vel:.6f}")

In [ ]:
# ── POSITIONS paradigm (baseline): u_t -> u_{t+1} ──
map_flat = lot_maps.reshape(T, R * d)  # (T, R*2)
pos_input = map_flat[:-1]   # u_0, ..., u_{T-2}
pos_target = map_flat[1:]   # u_1, ..., u_{T-1}

train_input_pos = pos_input[:T_TRAIN]
train_target_pos = pos_target[:T_TRAIN]

print(f"POSITIONS training: input {train_input_pos.shape}, target {train_target_pos.shape}")

rc_pos = ReservoirComputer(ReservoirConfig(
    input_size=R * d,
    reservoir_size=N,
    output_size=R * d,
    spectral_radius=PAPER_SR,
    input_scaling=PAPER_INPUT_SCALE,
    leak_rate=PAPER_LEAK,
    ridge_param=PAPER_RIDGE_RAW,  # 1e-4 for positions
    bias_scale=0.0,
    activation="tanh",
    init_scheme=InitScheme.SPARSE,
    sparsity=0.1,
    random_seed=42,
    use_operator_norm=False,
))

states_pos = rc_pos.run(train_input_pos)
rc_pos.train(states_pos, train_target_pos)

train_pred_pos = rc_pos.predict(states_pos)
train_rmse_pos = np.sqrt(np.mean((train_pred_pos - train_target_pos) ** 2))
print(f"  Train RMSE (positions): {train_rmse_pos:.6f}")

---
## Stage 4: Autonomous Forecasting (Closed Loop)

After T_train steps, the RC switches to closed-loop mode:
1. Predict Δ̂ = W_out^T [r; 1]
2. Feed Δ̂ back as input to advance reservoir state
3. Repeat for H=48 steps

**VELOCITY**: predicts velocities, which must be integrated later.

**POSITIONS**: directly predicts the next LOT map (no integration).

In [ ]:
# ── VELOCITY autonomous rollout ──
_, pred_vel_flat = rc_vel.run_autonomous(train_input_vel, H)
pred_vel = pred_vel_flat.reshape(H, R, d)
print(f"Predicted velocities: {pred_vel.shape}")

# ── POSITIONS autonomous rollout ──
_, pred_pos_flat = rc_pos.run_autonomous(train_input_pos, H)
pred_maps_pos = pred_pos_flat.reshape(H, R, d)
print(f"Predicted maps (positions): {pred_maps_pos.shape}")

---
## Stage 5: Integration & Reconstruction

**VELOCITY**: integrate predicted velocities to recover LOT maps:
$$\hat{u}_{T+k} = u_T + \sum_{j=1}^{k} \hat{\Delta}_{T+j}$$

The seed map is lot_maps[T_TRAIN] (the last known true LOT map at forecast start).

**POSITIONS**: the predicted maps ARE the forecast (no integration needed).

**Reconstruction**: The LOT map values u_hat(x_i) ARE the predicted particle positions — the pushforward of σ through the map gives exactly these points.

In [ ]:
# ── VELOCITY: cumulative sum integration ──
# Seed = last known LOT map (at the end of the training window)
# The velocity training used vel_input[:T_TRAIN] = vel_flat[0:T_TRAIN]
# which corresponds to velocities[0:T_TRAIN], i.e., maps[0:T_TRAIN+1].
# So the forecast starts at map index T_TRAIN.
seed_map = lot_maps[T_TRAIN]  # (R, d)

pred_maps_vel = np.empty((H + 1, R, d))
pred_maps_vel[0] = seed_map
for k in range(H):
    pred_maps_vel[k + 1] = pred_maps_vel[k] + pred_vel[k]

# The predicted measures are pred_maps_vel[1:] (H steps)
forecast_vel = pred_maps_vel[1:]  # (H, R, d)

# ── POSITIONS: already have the maps ──
forecast_pos = pred_maps_pos  # (H, R, d)

# ── Ground truth for comparison ──
true_maps = lot_maps[T_TRAIN + 1 : T_TRAIN + 1 + H]  # (H, R, d)
H_actual = min(H, true_maps.shape[0])  # may have fewer frames

print(f"Forecast velocity: {forecast_vel.shape}")
print(f"Forecast positions: {forecast_pos.shape}")
print(f"Ground truth: {true_maps.shape}")
print(f"Usable forecast horizon: {H_actual} steps")

---
## Stage 6: Evaluation

**L²(σ) RMSE** (paper §6):
$$\text{RMSE} = \sqrt{\frac{1}{H} \sum_{t=1}^{H} \frac{1}{R} \sum_{i=1}^{R} \|\hat{u}_t(x_i) - u_t(x_i)\|^2}$$

This is the primary metric. Lower = better. Velocity should be lower than positions.

In [ ]:
def l2_sigma_rmse(pred, true):
    """L²(σ) RMSE: sqrt(mean over time and ref points of ||pred - true||²)"""
    diff = pred[:H_actual] - true[:H_actual]
    sq_norms = np.sum(diff ** 2, axis=-1)   # (H, R)
    per_time = np.mean(sq_norms, axis=-1)    # (H,)
    return float(np.sqrt(np.mean(per_time)))

def l2_sigma_per_step(pred, true):
    """Per-step L²(σ) error."""
    diff = pred[:H_actual] - true[:H_actual]
    sq_norms = np.sum(diff ** 2, axis=-1)
    return np.sqrt(np.mean(sq_norms, axis=-1))

rmse_vel = l2_sigma_rmse(forecast_vel, true_maps)
rmse_pos = l2_sigma_rmse(forecast_pos, true_maps)
improvement = (rmse_pos - rmse_vel) / rmse_pos * 100

print(f"{'Method':<15} {'L²(σ) RMSE':>12}")
print(f"{'-'*30}")
print(f"{'Velocity':<15} {rmse_vel:12.6f}")
print(f"{'Positions':<15} {rmse_pos:12.6f}")
print(f"")
winner = "VELOCITY" if rmse_vel < rmse_pos else "POSITIONS"
print(f"Winner: {winner} ({improvement:+.1f}% improvement)")

In [ ]:
# Per-step error curves
err_vel = l2_sigma_per_step(forecast_vel, true_maps)
err_pos = l2_sigma_per_step(forecast_pos, true_maps)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(err_vel, color="#2166ac", lw=2, label=f"Velocity (RMSE={rmse_vel:.4f})")
ax.plot(err_pos, color="#b2182b", lw=2, label=f"Positions (RMSE={rmse_pos:.4f})")
ax.fill_between(range(H_actual), err_vel, err_pos, alpha=0.15,
                color="blue" if rmse_vel < rmse_pos else "red")
ax.set_xlabel("Forecast step"); ax.set_ylabel("L²(σ) error")
ax.set_title(f"{system_name}: Per-step error — {winner} wins ({improvement:+.1f}%)")
ax.legend(); ax.grid(alpha=0.2)
plt.show()

In [ ]:
# Three-row particle snapshot comparison
n_snaps = 6
snap_idx = [min(int(f * H_actual), H_actual - 1) for f in np.linspace(0, 0.9, n_snaps)]

fig, axes = plt.subplots(3, n_snaps, figsize=(3.5 * n_snaps, 10))
for col, h in enumerate(snap_idx):
    for row, (data, color, label) in enumerate([
        (true_maps[h], "#333333", "Ground truth"),
        (forecast_vel[h], "#2166ac", "Velocity pred."),
        (forecast_pos[h], "#b2182b", "Positions pred."),
    ]):
        ax = axes[row, col]
        ax.scatter(data[:, 0], data[:, 1], s=8, alpha=0.6, c=color, edgecolors="none")
        if row > 0:
            err = np.sqrt(np.mean(np.sum((data - true_maps[h]) ** 2, axis=-1)))
            ax.set_title(f"err = {err:.4f}", fontsize=8, color=color)
        else:
            ax.set_title(f"t = T+{h}", fontsize=10)
        if col == 0:
            ax.set_ylabel(label, fontsize=11, fontweight="bold",
                          color=color if row > 0 else "black")
        ax.set_aspect("equal"); ax.grid(alpha=0.15); ax.tick_params(labelsize=7)

fig.suptitle(f"{system_name}: Ground Truth / Velocity / Positions", fontsize=13)
fig.tight_layout()
plt.show()